In [1]:
!pip install langchain
!pip install scikit-learn
!pip install langchain-text-splitters

In [2]:
import requests
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

In [3]:
url = "https://en.wikipedia.org/wiki/Natural_language_processing"
text = requests.get(url).text

print("Document Length:", len(text))

Document Length: 126


In [4]:
def fixed_chunking(text, chunk_size):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

In [5]:
def recursive_chunking(text, chunk_size, overlap):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap
    )
    return splitter.split_text(text)

In [6]:
def retrieve_best_chunk(chunks, query):
    vectorizer = TfidfVectorizer()
    vectors = vectorizer.fit_transform(chunks + [query])

    chunk_vectors = vectors[:-1]
    query_vector = vectors[-1]

    similarities = cosine_similarity(query_vector, chunk_vectors)[0]
    best_idx = np.argmax(similarities)

    return best_idx, chunks[best_idx], similarities[best_idx]

In [7]:
chunk_sizes = [100, 300, 500, 1000]
overlaps = [0, 50, 100]

query = "What is natural language processing used for?"

results = []

In [8]:
for size in chunk_sizes:
    for overlap in overlaps:

        if overlap == 0:
            chunks = fixed_chunking(text, size)
            method = "Fixed"
        else:
            chunks = recursive_chunking(text, size, overlap)
            method = "Recursive"

        idx, best_chunk, score = retrieve_best_chunk(chunks, query)

        results.append({
            "method": method,
            "chunk_size": size,
            "overlap": overlap,
            "score": score,
            "chunk": best_chunk[:300]  # preview
        })

        print(f"\n--- {method} | Size: {size} | Overlap: {overlap} ---")
        print("Score:", score)
        print("Chunk Preview:", best_chunk[:200])


--- Fixed | Size: 100 | Overlap: 0 ---
Score: 0.0
Chunk Preview: Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricat

--- Recursive | Size: 100 | Overlap: 50 ---
Score: 0.0
Chunk Preview: Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also

--- Recursive | Size: 100 | Overlap: 100 ---
Score: 0.0
Chunk Preview: Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also

--- Fixed | Size: 300 | Overlap: 0 ---
Score: 0.0
Chunk Preview: Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.


--- Recursive | Size: 300 | Overlap: 50 ---
Score: 0.0
Chunk Preview: Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricator.wikimedia.org/T400119.

--- Recursive | Size: 300 | Overlap: 100 ---
Score: 0.0
Chunk Preview: Please set a user-agent and respect our robot policy https://w

In [9]:
best = max(results, key=lambda x: x["score"])

print("\n🏆 BEST CONFIGURATION:")
print(best)


🏆 BEST CONFIGURATION:
{'method': 'Fixed', 'chunk_size': 100, 'overlap': 0, 'score': np.float64(0.0), 'chunk': 'Please set a user-agent and respect our robot policy https://w.wiki/4wJS. See also https://phabricat'}


In [10]:
def find_boundary_issues(text, chunk_size):
    issues = []
    for i in range(chunk_size, len(text), chunk_size):
        snippet = text[i-50:i+50]
        if "." not in text[i-5:i+5]:  # likely mid-sentence split
            issues.append((i, snippet))
    return issues

issues = find_boundary_issues(text, 300)

print("\n🚨 Boundary Issues Examples:")
for i, (pos, snippet) in enumerate(issues[:3]):
    print(f"\nExample {i+1}")
    print("Position:", pos)
    print("Snippet:", snippet)


🚨 Boundary Issues Examples:


In [11]:
print("\n📌 RECOMMENDATION:")
print("Best configuration based on similarity score:")
print(f"Chunk Size: {best['chunk_size']}, Overlap: {best['overlap']}")
print("Reason: Highest semantic similarity with query and better context preservation.")


📌 RECOMMENDATION:
Best configuration based on similarity score:
Chunk Size: 100, Overlap: 0
Reason: Highest semantic similarity with query and better context preservation.
